# 복합 양자 시스템 (HilbertSpace)

이 노트북에서는 `HilbertSpace`를 사용하여 여러 양자 서브시스템을 결합하고 분석합니다.

**다루는 내용:**
- Part 1: Kerr 진동자 쌍의 빔스플리터 결합
- Bare vs Dressed 스펙트럼
- Bare/Dressed 상태 매핑
- Dressed 고유 기저에서의 연산자
- Part 2: Transmon + Oscillator (분산 읽기)

In [ ]:
using ScQubitsMimic
using CairoMakie
using LinearAlgebra

## 텐서곱 힐베르트 공간

복합 양자 시스템의 힐베르트 공간은 각 서브시스템의 텐서곱으로 구성됩니다:

$$\mathcal{H} = \mathcal{H}_1 \otimes \mathcal{H}_2 \otimes \cdots$$

전체 해밀토니안은 각 서브시스템의 해밀토니안과 상호작용 항의 합입니다:

$$H = \sum_i H_i \otimes \mathbb{1}_{\bar{i}} + \sum_{i<j} H_{\text{int},ij}$$

---
## Part 1: Kerr 진동자 결합

두 개의 Kerr 진동자를 빔스플리터 상호작용으로 결합합니다.

**Kerr 진동자 해밀토니안:**
$$H_K = E_{\text{osc}} \hat{a}^\dagger\hat{a} + \frac{K}{2} \hat{a}^\dagger\hat{a}(\hat{a}^\dagger\hat{a} - 1)$$

**빔스플리터 상호작용:**
$$H_{\text{int}} = g(\hat{a}_1^\dagger \hat{a}_2 + \hat{a}_1 \hat{a}_2^\dagger)$$

In [ ]:
kosc1 = KerrOscillator(E_osc=4.5, K=-0.05, truncated_dim=8)
kosc2 = KerrOscillator(E_osc=6.0, K=-0.03, truncated_dim=8)

println("서브시스템 1: KerrOscillator, ω=$(kosc1.E_osc) GHz, K=$(kosc1.K) GHz")
println("서브시스템 2: KerrOscillator, ω=$(kosc2.E_osc) GHz, K=$(kosc2.K) GHz")

# HilbertSpace 생성 및 상호작용 추가
hs = HilbertSpace([kosc1, kosc2])
g = 0.1  # GHz
add_interaction!(hs, g, [kosc1, kosc2],
    [s -> creation_operator(s), s -> annihilation_operator(s)])
add_interaction!(hs, g, [kosc1, kosc2],
    [s -> annihilation_operator(s), s -> creation_operator(s)])

println("\n빔스플리터 결합: g = $g GHz")
println("전체 힐베르트 공간 차원: $(hilbertdim(hs))")

### Bare vs Dressed 스펙트럼

- **Bare 스펙트럼**: 상호작용이 없는 상태의 고유 에너지
- **Dressed 스펙트럼**: 상호작용을 포함한 전체 해밀토니안의 고유 에너지

결합에 의한 에너지 이동을 비교합니다.

In [ ]:
# Bare 스펙트럼 (상호작용 없음)
hs_bare = HilbertSpace([kosc1, kosc2])
bare_vals = eigenvals(hs_bare; evals_count=10)

# Dressed 스펙트럼 (상호작용 포함)
dressed_vals = eigenvals(hs; evals_count=10)

println("  준위    Bare (GHz)      Dressed (GHz)   차이 (MHz)")
println("  " * "-"^55)
for i in 1:10
    diff = (dressed_vals[i] - bare_vals[i]) * 1000
    println("  E_$(i-1)     $(round(bare_vals[i], digits=4))        $(round(dressed_vals[i], digits=4))        $(round(diff, digits=2))")
end

In [ ]:
# 바 차트 비교
fig = Figure(size=(700, 400))
ax = Axis(fig[1, 1],
    xlabel="에너지 준위",
    ylabel="에너지 - E₀ (GHz)",
    title="Bare vs Dressed 스펙트럼",
    xticks=0:9)
barplot!(ax, collect(0:9) .- 0.15, bare_vals .- bare_vals[1],
    width=0.3, label="Bare", color=:steelblue)
barplot!(ax, collect(0:9) .+ 0.15, dressed_vals .- dressed_vals[1],
    width=0.3, label="Dressed", color=:coral)
axislegend(ax, position=:lt)
fig

### Bare/Dressed 상태 매핑

`generate_lookup!`을 사용하면 bare 상태와 dressed 상태 사이의 대응 관계를 자동으로 계산합니다.

- `dressed_index(hs, n1, n2)`: bare 인덱스 $(n_1, n_2)$에 대응하는 dressed 상태 번호
- `bare_index(hs, i)`: dressed 상태 $i$에 대응하는 bare 인덱스
- `energy_by_bare_index(hs, n1, n2)`: bare 상태의 dressed 에너지

In [ ]:
lookup = generate_lookup!(hs; evals_count=10)

println("Dressed → Bare 상태 매핑:")
for i in 1:10
    bi = bare_index(hs, i)
    e = energy_by_dressed_index(hs, i)
    println("  dressed[$i] → |$(bi[1]-1),$(bi[2]-1)⟩, E = $(round(e, digits=6)) GHz")
end

In [ ]:
# 특정 bare 상태 조회
println("특정 Bare 상태의 Dressed 에너지:")
states_to_find = [(1,1), (2,1), (1,2), (2,2), (3,1), (1,3)]
for (n1, n2) in states_to_find
    try
        di = dressed_index(hs, n1, n2)
        e = energy_by_bare_index(hs, n1, n2)
        println("  |$(n1-1),$(n2-1)⟩ → dressed[$di], E = $(round(e, digits=6)) GHz")
    catch ex
        println("  |$(n1-1),$(n2-1)⟩ → 조회 범위 밖")
    end
end

### Dressed 고유 기저에서의 연산자

`op_in_dressed_eigenbasis`를 사용하면 임의의 연산자를 dressed 고유 기저로 변환할 수 있습니다. 이는 실험적으로 측정 가능한 물리량을 계산하는 데 유용합니다.

In [ ]:
# 수 연산자 n̂₁을 전체 공간으로 확장
dims = [hilbertdim(s) for s in hs.subsystems]
n1_full = identity_wrap(number_operator(kosc1), 1, dims)

# Dressed 고유 기저로 변환
n1_dressed = op_in_dressed_eigenbasis(hs, n1_full; truncated_dim=5)

println("⟨dressed_i|n̂₁|dressed_j⟩ (5×5):")
for i in 1:5
    row = ["$(round(real(n1_dressed[i,j]), digits=4))" for j in 1:5]
    println("  ", join(row, "  "))
end

---
## Part 2: Transmon + Oscillator (분산 읽기)

Transmon 큐비트를 공진기(resonator)에 결합하는 것은 회로 양자 전기역학(circuit QED)의 핵심 구성입니다.

**Jaynes-Cummings 형태의 결합:**
$$H_{\text{int}} = g \, \hat{n} \otimes (\hat{a} + \hat{a}^\dagger)$$

분산 영역($|\omega_q - \omega_r| \gg g$)에서 **분산 시프트** $\chi$는 큐비트 상태에 따른 공진기 주파수 변화를 나타냅니다:

$$\chi = (E_{|1,1\rangle} - E_{|1,0\rangle}) - (E_{|0,1\rangle} - E_{|0,0\rangle})$$

In [ ]:
tmon = Transmon(EJ=30.0, EC=1.2, ng=0.0, ncut=15, truncated_dim=6)
res = Oscillator(E_osc=7.0, truncated_dim=10)

hs2 = HilbertSpace([tmon, res])

# Jaynes-Cummings 결합
g_jc = 0.2
add_interaction!(hs2, g_jc, [tmon, res],
    [s -> n_operator(s), s -> annihilation_operator(s) + creation_operator(s)])

println("Transmon: EJ=$(tmon.EJ), EC=$(tmon.EC) GHz")
println("공진기: ω_r=$(res.E_osc) GHz")
println("결합 상수: g=$(g_jc) GHz")
println("전체 차원: $(hilbertdim(hs2))")

In [ ]:
lookup2 = generate_lookup!(hs2; evals_count=15)

println("Dressed 스펙트럼 (상태 할당 포함):")
for i in 1:15
    bi = bare_index(hs2, i)
    e = energy_by_dressed_index(hs2, i)
    println("  dressed[$i] → |q=$(bi[1]-1), n=$(bi[2]-1)⟩, E = $(round(e, digits=4)) GHz")
end

In [ ]:
# 분산 시프트 계산
try
    E00 = energy_by_bare_index(hs2, 1, 1)  # |g,0⟩
    E10 = energy_by_bare_index(hs2, 2, 1)  # |e,0⟩
    E01 = energy_by_bare_index(hs2, 1, 2)  # |g,1⟩
    E11 = energy_by_bare_index(hs2, 2, 2)  # |e,1⟩

    χ = (E11 - E10) - (E01 - E00)
    println("분산 시프트:")
    println("  E_{|g,0⟩} = $(round(E00, digits=4)) GHz")
    println("  E_{|e,0⟩} = $(round(E10, digits=4)) GHz")
    println("  E_{|g,1⟩} = $(round(E01, digits=4)) GHz")
    println("  E_{|e,1⟩} = $(round(E11, digits=4)) GHz")
    println("\n  χ/2π = $(round(χ * 1000, digits=2)) MHz")
catch ex
    println("분산 시프트를 계산할 수 없습니다: $ex")
end